# Flujo de posesión · mapa de viento por equipo

Por dónde mueve el balón cada equipo en su temporada, dibujado como un mapa de viento: las rutas que más usa
son corrientes gruesas y brillantes, las zonas que casi no pisa quedan en calma y las partículas van más rápido
donde el balón va más rápido. Idea: el blueprint
[possession flow](https://github.com/opengoalapp/football-blueprints/tree/main/possession-flow) de opengoalapp.

Es el mismo cálculo que la sección **Flujo de posesión** de FOS Scout Lab (`lib/flujoPosesion.ts` y el puente),
en un notebook para correrlo en tu equipo:

1. Baja los eventos de StatsBomb de todos los partidos jugados de la liga (la primera vez tarda unos minutos;
   después salen de la caché, la misma que usa el puente de la app).
2. Calcula el **promedio por partido de cada equipo**: balón recorrido, sentido, carriles y velocidad por tercio,
   contra la media de la liga.
3. Dibuja el mapa de todos los equipos y una animación de dos equipos lado a lado.

**Credenciales.** No se escriben en el notebook. Se leen, por este orden, de `SB_USERNAME` / `SB_PASSWORD`,
del Llavero de macOS (la entrada `fos-scouting` / `statsbomb` que ya usa la app) o de
`~/.fos-scouting/credentials.json`; si no hay ninguna, el notebook las pide con un campo oculto. Córrelo en
Jupyter o VS Code en tu equipo, no en Colab, para que no salgan de él.

Paquetes: `requests numpy pandas matplotlib pillow` (en una terminal: `pip install requests numpy pandas matplotlib pillow`).

## 1 · Qué liga y qué equipos

In [ ]:
from pathlib import Path

LIGA = "Canadian Premier League"
TEMPORADA = None          # None = la más reciente con partidos; o "2026", "2025", …
EQUIPO_A = "Cavalry"      # cancha izquierda de la animación (basta con parte del nombre)
EQUIPO_B = "Forge"        # cancha derecha

DETALLE = 0.5             # 0–1: cuánto flujo secundario se ve (umbral de ruta y bifurcaciones)
ANCHO = 0.3               # 0–1: ríos estrechos (0) o corrientes anchas (1)
ESTELA = 0.5              # 0–1: largo de la estela de cada partícula

FUENTE = "api"            # "api": tu cuenta de StatsBomb · "abierta": datos abiertos, para probar sin cuenta
                          #   (con "abierta": LIGA = "FA Women's Super League", TEMPORADA = "2023/2024")
CACHE = Path.home() / ".fos-scouting" / "sb-events-cache"   # la misma caché que el puente de la app
SALIDA = Path("flujo_posesion")                              # tablas, imágenes y la animación

## 2 · Credenciales y descarga

In [ ]:
import getpass
import json
import os
import subprocess
import sys
import unicodedata
from concurrent.futures import ThreadPoolExecutor

import requests


def credenciales():
    """Usuario y contraseña de StatsBomb sin escribirlos nunca en el notebook."""
    usuario = os.environ.get("SB_USERNAME") or os.environ.get("STATSBOMB_USERNAME")
    clave = os.environ.get("SB_PASSWORD") or os.environ.get("STATSBOMB_PASSWORD")
    if usuario and clave:
        return usuario, clave, "variables de entorno"
    if sys.platform == "darwin":
        try:
            salida = subprocess.run(
                ["/usr/bin/security", "find-generic-password", "-s", "fos-scouting", "-a", "statsbomb", "-w"],
                capture_output=True, text=True, timeout=5,
            )
            usuario, _, clave = salida.stdout.strip().partition(":")
            if salida.returncode == 0 and usuario and clave:
                return usuario, clave, "Llavero de macOS"
        except Exception:
            pass
    try:
        guardadas = json.loads((Path.home() / ".fos-scouting" / "credentials.json").read_text()).get("statsbomb", {})
        if guardadas.get("username") and guardadas.get("password"):
            return guardadas["username"], guardadas["password"], "~/.fos-scouting/credentials.json"
    except Exception:
        pass
    usuario = input("Usuario de StatsBomb: ").strip()
    clave = getpass.getpass("Contraseña de StatsBomb: ")
    return usuario, clave, "escritas a mano (no se guardan)"


if FUENTE == "api":
    _usuario, _clave, _origen = credenciales()
    AUTH = (_usuario, _clave)
    del _clave
    print(f"Credenciales de StatsBomb: {_origen}.")
else:
    AUTH = None
    CACHE = SALIDA / "cache-abierta"
    print("Datos abiertos de StatsBomb: no hacen falta credenciales.")

API = "https://data.statsbomb.com/api"
ABIERTA = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"


def pedir(url):
    # Sin sesión válida StatsBomb no responde 401: redirige a su página de login.
    respuesta = requests.get(url, auth=AUTH, timeout=120, allow_redirects=False)
    if respuesta.status_code in (301, 302, 303, 307, 308, 401, 403):
        raise PermissionError(f"StatsBomb no aceptó las credenciales ({respuesta.status_code}): revisa usuario y contraseña.")
    respuesta.raise_for_status()
    return respuesta.json()


def sin_tildes(texto):
    texto = unicodedata.normalize("NFD", str(texto or ""))
    return "".join(c for c in texto if unicodedata.category(c) != "Mn").lower().strip()


catalogo = pedir(f"{API}/v4/competitions" if FUENTE == "api" else f"{ABIERTA}/competitions.json")
candidatas = [c for c in catalogo if sin_tildes(c.get("competition_name")) == sin_tildes(LIGA)]
if not candidatas:
    ligas = sorted({c.get("competition_name") for c in catalogo})
    raise ValueError(f"No encuentro «{LIGA}». Ligas disponibles: {', '.join(map(str, ligas))}")
if TEMPORADA:
    candidatas = [c for c in candidatas if str(TEMPORADA) in str(c.get("season_name"))]
    if not candidatas:
        raise ValueError(f"«{LIGA}» no tiene la temporada {TEMPORADA}.")
con_partidos = [c for c in candidatas if c.get("match_available")] or candidatas
competicion = max(con_partidos, key=lambda c: str(c.get("season_name")))
CID, SID = competicion["competition_id"], competicion["season_id"]
NOMBRE_LIGA = f"{competicion['competition_name']} {competicion['season_name']}"

partidos = pedir(f"{API}/v6/competitions/{CID}/seasons/{SID}/matches" if FUENTE == "api"
                 else f"{ABIERTA}/matches/{CID}/{SID}.json")
jugados = sorted((p for p in partidos if p.get("home_score") is not None), key=lambda p: str(p.get("match_date")))
print(f"{NOMBRE_LIGA}: {len(jugados)} partidos jugados.")

In [ ]:
# Los tramos de cada partido: pases y conducciones del equipo con el balón,
# [equipo, x, y, xf, yf, segundos]. Sin saques de esquina ni de centro: salen
# siempre del mismo sitio y dibujarían corrientes que el equipo no elige.
# Se guardan en el mismo formato que el puente (pf-<partido>.json), así que la
# app y el notebook comparten la descarga.
VERSION_TRAMOS = 1
SIN_PASE = {"Corner", "Kick Off"}


def extraer_tramos(eventos):
    tramos = []
    for e in eventos:
        tipo = (e.get("type") or {}).get("name")
        if tipo not in ("Pass", "Carry"):
            continue
        equipo = (e.get("team") or {}).get("id")
        if equipo is None or equipo != (e.get("possession_team") or {}).get("id"):
            continue
        if tipo == "Pass":
            pase = e.get("pass") or {}
            if (pase.get("type") or {}).get("name") in SIN_PASE:
                continue
            fin = pase.get("end_location")
        else:
            fin = (e.get("carry") or {}).get("end_location")
        inicio = e.get("location")
        if not inicio or not fin or len(inicio) < 2 or len(fin) < 2:
            continue
        duracion = e.get("duration")
        tramos.append([equipo, round(inicio[0], 1), round(inicio[1], 1), round(fin[0], 1), round(fin[1], 1),
                       round(duracion, 2) if isinstance(duracion, (int, float)) else None])
    return tramos


def tramos_del_partido(match_id):
    ruta = CACHE / f"pf-{int(match_id)}.json"
    if ruta.exists():
        try:
            guardado = json.loads(ruta.read_text(encoding="utf-8"))
            if guardado.get("v") == VERSION_TRAMOS:
                return guardado.get("tramos") or []
        except Exception:
            pass
    try:
        eventos = pedir(f"{API}/v8/events/{int(match_id)}" if FUENTE == "api" else f"{ABIERTA}/events/{int(match_id)}.json")
    except Exception as fallo:
        print(f"  partido {match_id}: {fallo}")
        return None
    tramos = extraer_tramos(eventos)
    temporal = ruta.with_name(f"{ruta.name}.{os.getpid()}.tmp")
    temporal.write_text(json.dumps({"v": VERSION_TRAMOS, "tramos": tramos}), encoding="utf-8")
    os.replace(temporal, ruta)
    return tramos


CACHE.mkdir(parents=True, exist_ok=True)
SALIDA.mkdir(parents=True, exist_ok=True)
pendientes = sum(not (CACHE / f"pf-{int(p['match_id'])}.json").exists() for p in jugados)
print(f"Leyendo {len(jugados)} partidos ({pendientes} sin caché; StatsBomb tarda unos segundos por partido)…")
with ThreadPoolExecutor(max_workers=6) as grupo:
    por_partido = list(grupo.map(lambda p: tramos_del_partido(p["match_id"]), jugados))
fallidos = sum(t is None for t in por_partido)
print(f"Listo: {len(jugados) - fallidos} partidos" + (f", {fallidos} sin poder leer (vuelve a correr la celda)." if fallidos else "."))

## 3 · La rejilla y las corrientes

Cada tramo se reparte por una rejilla de 60 × 40 celdas y 16 direcciones **a lo largo de su recorrido**: un pase
largo suma un poco en cada celda que cruza, no todo en la de salida. En cada celda se guardan además la distancia
y el tiempo del balón, para la velocidad (pausas incluidas).

Para que no salga una maraña: en cada zona se dibuja la **dirección dominante** (y una segunda solo si el reparto
es parejo; las direcciones nunca se promedian), y qué cuenta como ruta se decide **contra toda la liga**, así que
el grosor significa lo mismo en todos los equipos.

In [ ]:
import numpy as np

COLS, FILAS, DIRS, CELDA, PASO = 60, 40, 16, 2.0, 1.0
METROS = 0.9144                     # las coordenadas de StatsBomb son yardas (120 × 80)

nombres, partidos_equipo, tramos_equipo = {}, {}, {}
for partido, lista in zip(jugados, por_partido):
    if lista is None:
        continue
    for lado in ("home_team", "away_team"):
        equipo = partido.get(lado) or {}
        ident = equipo.get(f"{lado}_id")
        nombres[ident] = equipo.get(f"{lado}_name") or str(ident)
        partidos_equipo[ident] = partidos_equipo.get(ident, 0) + 1
    for tramo in lista:
        tramos_equipo.setdefault(tramo[0], []).append(tramo[1:])


def rasterizar(tramos):
    """flujo[celda, dirección], recorrido[celda] y tiempo[celda] de una temporada."""
    flujo = np.zeros(COLS * FILAS * DIRS)
    recorrido = np.zeros(COLS * FILAS)
    tiempo = np.zeros(COLS * FILAS)
    if not tramos:
        return flujo, recorrido, tiempo
    t = np.array([[np.nan if v is None else v for v in tramo] for tramo in tramos], dtype=float)
    x, y, xf, yf, seg = t.T
    dx, dy = xf - x, yf - y
    largo = np.hypot(dx, dy)
    with np.errstate(divide="ignore", invalid="ignore"):
        con_tiempo = np.isfinite(seg) & (seg > 0) & (seg <= 20) & (largo / seg <= 45)

    def celda(px, py):
        c = np.clip((px / CELDA).astype(int), 0, COLS - 1)
        f = np.clip((py / CELDA).astype(int), 0, FILAS - 1)
        return f * COLS + c

    # Una conducción casi parada suma segundos y ninguna distancia: la zona donde se pausa sale lenta.
    quieto = largo < 0.5
    np.add.at(tiempo, celda(x[quieto & con_tiempo], y[quieto & con_tiempo]), seg[quieto & con_tiempo])

    m = ~quieto
    x, y, dx, dy, largo, seg, con_tiempo = x[m], y[m], dx[m], dy[m], largo[m], seg[m], con_tiempo[m]
    direccion = np.round(np.arctan2(dy, dx) / (2 * np.pi / DIRS)).astype(int) % DIRS
    n = np.maximum(1, np.ceil(largo / PASO).astype(int))
    cual = np.repeat(np.arange(len(n)), n)
    k = np.arange(len(cual)) - np.repeat(np.cumsum(n) - n, n)
    avance = (k + 0.5) / n[cual]
    c = celda(x[cual] + dx[cual] * avance, y[cual] + dy[cual] * avance)
    paso = (largo / n)[cual]
    np.add.at(flujo, c * DIRS + direccion[cual], paso)
    v = con_tiempo[cual]
    np.add.at(recorrido, c[v], paso[v])
    np.add.at(tiempo, c[v], (seg / n)[cual][v])
    return flujo, recorrido, tiempo


equipos = []
for ident in sorted(partidos_equipo, key=lambda i: sin_tildes(nombres[i])):
    flujo, recorrido, tiempo = rasterizar(tramos_equipo.get(ident, []))
    equipos.append({"id": ident, "nombre": nombres[ident], "partidos": partidos_equipo[ident],
                    "flujo": flujo, "recorrido": recorrido, "tiempo": tiempo})
print(f"{len(equipos)} equipos: " + ", ".join(e["nombre"] for e in equipos))

In [ ]:
def desenfocar(a, sigma, ejes):
    """Desenfoque gaussiano; en los bordes se renormaliza para no apagar la orilla."""
    if sigma <= 0:
        return a.copy()
    r = max(1, int(np.ceil(sigma * 2.5)))
    nucleo = np.exp(-(np.arange(-r, r + 1) ** 2) / (2 * sigma * sigma))
    for eje in ejes:
        suma = np.zeros_like(a)
        pesos = np.zeros(a.shape[eje])
        largo = a.shape[eje]
        for j, w in zip(range(-r, r + 1), nucleo):
            origen = slice(max(0, j), largo + min(0, j))
            hacia = slice(max(0, -j), largo - max(0, j))
            idx_o = [slice(None)] * a.ndim
            idx_h = [slice(None)] * a.ndim
            idx_o[eje], idx_h[eje] = origen, hacia
            suma[tuple(idx_h)] += w * a[tuple(idx_o)]
            pesos[hacia] += w
        forma = [1] * a.ndim
        forma[eje] = largo
        a = suma / pesos.reshape(forma)
    return a


ANGULOS = np.arange(DIRS) * 2 * np.pi / DIRS
COS, SEN = np.cos(ANGULOS), np.sin(ANGULOS)


def corrientes_en_bruto(equipo, sigma):
    """Por celda: la corriente principal (sector con más balón y sus vecinos) y la segunda candidata."""
    v = equipo["flujo"].reshape(FILAS, COLS, DIRS) / max(1, equipo["partidos"])
    v = desenfocar(v, sigma, (1, 0)).reshape(-1, DIRS)
    s = 0.25 * np.roll(v, 1, axis=1) + 0.5 * v + 0.25 * np.roll(v, -1, axis=1)
    filas = np.arange(len(v))
    p1 = np.argmax(s, axis=1)
    ventana = lambda p: (p[:, None] + np.array([-1, 0, 1])[None, :]) % DIRS
    v1 = np.take_along_axis(v, ventana(p1), axis=1)
    f1 = v1.sum(1)
    x1, y1 = (v1 * COS[ventana(p1)]).sum(1), (v1 * SEN[ventana(p1)]).sum(1)
    l1 = np.hypot(x1, y1)
    l1[l1 == 0] = 1
    distancia = np.abs(np.arange(DIRS)[None, :] - p1[:, None]) % DIRS
    distancia = np.minimum(distancia, DIRS - distancia)
    pico = (s >= np.roll(s, 1, axis=1)) & (s >= np.roll(s, -1, axis=1)) & (s > 0) & (distancia >= 3)
    p2 = np.argmax(np.where(pico, s, -np.inf), axis=1)
    hay2 = pico.any(1) & (s[filas, p1] > 0)
    v2 = np.take_along_axis(v, ventana(p2), axis=1)
    f2 = np.where(hay2, v2.sum(1), 0)
    x2, y2 = (v2 * COS[ventana(p2)]).sum(1), (v2 * SEN[ventana(p2)]).sum(1)
    l2 = np.hypot(x2, y2)
    l2[l2 == 0] = 1
    with np.errstate(divide="ignore", invalid="ignore"):
        relacion = np.where(hay2, s[filas, p2] / s[filas, p1], 0)
    vacia = s[filas, p1] <= 0
    f1[vacia] = 0
    return {"f1": f1, "ux1": x1 / l1, "uy1": y1 / l1, "f2": f2, "ux2": x2 / l2, "uy2": y2 / l2, "relacion": relacion}


def velocidad_por_celda(equipo):
    por_partido = 1 / max(1, equipo["partidos"])
    r = desenfocar(equipo["recorrido"].reshape(FILAS, COLS) * por_partido, 1.5, (1, 0)).ravel()
    t = desenfocar(equipo["tiempo"].reshape(FILAS, COLS) * por_partido, 1.5, (1, 0)).ravel()
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where((r >= 1) & (t > 0), r / t * METROS, np.nan)


def campos_de_liga(equipos, detalle=DETALLE, ancho=ANCHO):
    """Las corrientes de todos los equipos en una sola escala, la de la liga."""
    brutos = [corrientes_en_bruto(e, 0.55 + ancho * 1.6) for e in equipos]
    todas = np.sort(np.concatenate([b["f1"][b["f1"] > 0] for b in brutos]))
    umbral = np.quantile(todas, 0.8 - 0.4 * detalle)
    tope = max(umbral * 1.0001, np.quantile(todas, 0.985))
    escalar = lambda f: np.clip((f - umbral) / (tope - umbral), 0, 1) ** 0.8
    relacion_minima = 0.85 - 0.5 * detalle
    velocidades = [velocidad_por_celda(e) for e in equipos]
    campos = []
    for e, b, vel in zip(equipos, brutos, velocidades):
        fuerza1 = escalar(b["f1"])
        fuerza2 = np.where(b["relacion"] >= relacion_minima, escalar(b["f2"]), 0)
        campos.append({"id": e["id"], "nombre": e["nombre"], "fuerza": (fuerza1, fuerza2),
                       "ux": (b["ux1"], b["ux2"]), "uy": (b["uy1"], b["uy2"]),
                       "intensidad": np.maximum(fuerza1, fuerza2), "velocidad": vel, "metros": b["f1"] * METROS})
    con_ruta = np.sort(np.concatenate([c["velocidad"][(c["intensidad"] > 0) & np.isfinite(c["velocidad"])] for c in campos]))
    for c in campos:
        if len(con_ruta) > 1:
            rango = np.searchsorted(con_ruta, np.nan_to_num(c["velocidad"], nan=0), side="left") / (len(con_ruta) - 1)
        else:
            rango = np.full(COLS * FILAS, 0.5)
        c["rapidez"] = np.where(np.isfinite(c["velocidad"]), np.clip(rango, 0, 1), 0.5)
    pesos = np.array([(c["intensidad"] ** 2).sum() for c in campos])
    for c, peso in zip(campos, pesos):
        c["particulas"] = int(round(2600 * peso / max(pesos.max(), 1e-9)))
    escala = {"lenta": np.quantile(con_ruta, 0.1), "rapida": np.quantile(con_ruta, 0.9)}
    return campos, escala


campos, escala = campos_de_liga(equipos)
print(f"Velocidad del balón en las rutas: {escala['lenta']:.1f} m/s (lenta) a {escala['rapida']:.1f} m/s (rápida).")

## 4 · El promedio de cada equipo

Por partido y contra la media de la liga. **Hacia delante / atrás**: el recorrido del balón a menos de 56° de cada
portería. **Carriles**: reparto del recorrido según ataca el equipo (izquierda = banda de arriba en el mapa).
**Velocidad**: m/s del balón en cada tercio, pausas incluidas.

In [ ]:
import pandas as pd


def resumen(equipo):
    flujo = equipo["flujo"].reshape(FILAS, COLS, DIRS)
    total = flujo.sum()
    carril = np.minimum(2, ((np.arange(FILAS) + 0.5) * CELDA / (FILAS * CELDA) * 3).astype(int))
    tercio = np.minimum(2, ((np.arange(COLS) + 0.5) * CELDA / (COLS * CELDA) * 3).astype(int))
    limite = np.cos(np.pi / 4) - 1e-6
    por_carril = [flujo[carril == k].sum() / total for k in range(3)]
    rec = equipo["recorrido"].reshape(FILAS, COLS)
    tie = equipo["tiempo"].reshape(FILAS, COLS)
    vel = [rec[:, tercio == k].sum() / tie[:, tercio == k].sum() * METROS if tie[:, tercio == k].sum() > 0 else np.nan
           for k in range(3)]
    return {
        "Equipo": equipo["nombre"],
        "PJ": equipo["partidos"],
        "Balón por partido (m)": total / max(1, equipo["partidos"]) * METROS,
        "Hacia delante %": flujo[:, :, COS >= limite].sum() / total * 100,
        "Hacia atrás %": flujo[:, :, COS <= -limite].sum() / total * 100,
        "Izquierda %": por_carril[0] * 100,
        "Centro %": por_carril[1] * 100,
        "Derecha %": por_carril[2] * 100,
        "Vel. salida (m/s)": vel[0],
        "Vel. medio campo (m/s)": vel[1],
        "Vel. último tercio (m/s)": vel[2],
        "Vel. media (m/s)": rec.sum() / tie.sum() * METROS,
    }


tabla = pd.DataFrame([resumen(e) for e in equipos]).set_index("Equipo")
tabla = tabla.sort_values("Balón por partido (m)", ascending=False)
media = tabla.drop(columns="PJ").mean()
diferencias = (tabla.drop(columns="PJ") - media).add_suffix(" vs liga")

con_media = pd.concat([tabla, media.to_frame("Media de la liga").T])
formato = {c: "{:.0f}" for c in con_media.columns if c.endswith("(m)") or c.endswith("%")}
formato.update({c: "{:.1f}" for c in con_media.columns if "m/s" in c})
formato["PJ"] = "{:.0f}"
archivo = SALIDA / f"promedios_{''.join(c if c.isalnum() else '_' for c in sin_tildes(NOMBRE_LIGA)).strip('_')}.csv"
con_media.round(2).to_csv(archivo)
print(f"Guardado en {archivo}")
try:
    display(con_media.style.format(formato, na_rep="—"))
except NameError:  # fuera de Jupyter
    print(con_media.round(1).to_string())

In [ ]:
# Lo mismo, como diferencia con la media de la liga: + es más que la media (en pp para los %).
formato_dif = {c: "{:+.0f}" for c in diferencias.columns if "(m)" in c or "%" in c}
formato_dif.update({c: "{:+.1f}" for c in diferencias.columns if "m/s" in c})
try:
    display(diferencias.style.format(formato_dif, na_rep="—").background_gradient(cmap="BrBG", axis=0))
except NameError:
    print(diferencias.round(1).to_string())

## 5 · El mapa

Las partículas siguen las corrientes de cada equipo. **Grosor y cantidad** = cuánto balón pasa por la ruta, en la
escala de la liga. **Color** = velocidad del balón en la zona, de lenta (verde apagado) a rápida (casi blanco). Todos
los equipos atacan hacia la derecha; la banda de arriba es su izquierda.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgb
from matplotlib.patches import Arc, Rectangle

CAMPO = "#0e1512"
RAMPA = np.array([to_rgb(c) for c in ["#2c6b5a", "#26876b", "#16a57c", "#12c48b", "#4dd6a3", "#8be5c2", "#d2f6e6"]])
GROSORES = np.array([0.35, 0.65, 1.1])   # puntos, de la ruta ocasional a la principal


def dibujar_cancha(ax):
    ax.set_facecolor(CAMPO)
    linea = dict(color="white", alpha=0.78, lw=0.8, fill=False)
    ax.add_patch(Rectangle((0, 0), 120, 80, **linea))
    ax.plot([60, 60], [0, 80], color="white", alpha=0.78, lw=0.8)
    ax.add_patch(Arc((60, 40), 20, 20, **{k: v for k, v in linea.items() if k != "fill"}))
    apertura = np.degrees(np.arccos(0.6))
    for lado in (0, 1):
        x = (lambda v: 120 - v) if lado else (lambda v: v)
        ax.add_patch(Rectangle((min(x(0), x(18)), 18), 18, 44, **linea))
        ax.add_patch(Rectangle((min(x(0), x(6)), 30), 6, 20, **linea))
        ax.add_patch(Rectangle((min(x(0), x(-1.6)), 36), 1.6, 8, **linea))
        ax.scatter([x(12)], [40], s=3, color="white", alpha=0.78, zorder=3)
        ax.add_patch(Arc((x(12), 40), 20, 20, theta1=(180 - apertura) if lado else -apertura,
                         theta2=(180 + apertura) if lado else apertura, color="white", alpha=0.78, lw=0.8))
    ax.scatter([60], [40], s=3, color="white", alpha=0.78, zorder=3)
    ax.set_xlim(-3, 123)
    ax.set_ylim(83, -3)            # y = 0 arriba: la banda izquierda del que ataca
    ax.set_aspect("equal")
    ax.axis("off")


def bilineal(valores, x, y):
    gx = np.clip(x / CELDA - 0.5, 0, COLS - 1)
    gy = np.clip(y / CELDA - 0.5, 0, FILAS - 1)
    c0, f0 = np.floor(gx).astype(int), np.floor(gy).astype(int)
    c1, f1 = np.minimum(c0 + 1, COLS - 1), np.minimum(f0 + 1, FILAS - 1)
    tx, ty = gx - c0, gy - f0
    v = valores
    arriba = v[f0 * COLS + c0] * (1 - tx) + v[f0 * COLS + c1] * tx
    abajo = v[f1 * COLS + c0] * (1 - tx) + v[f1 * COLS + c1] * tx
    return arriba * (1 - ty) + abajo * ty


class Particulas:
    """Partículas que siguen su propia corriente, se ciñen al centro del canal y se apagan en la calma."""

    def __init__(self, campo, n=None, estela=ESTELA, ancho=ANCHO, semilla=7):
        self.c = campo
        self.rng = np.random.default_rng(semilla)
        self.n = campo["particulas"] if n is None else n
        self.l = int(round(6 + estela * 30))
        self.cenido = 0.6 + (1 - ancho) * 2.4
        peso = campo["intensidad"] ** 1.3
        self.prob = peso / peso.sum() if peso.sum() > 0 else None
        z = lambda: np.zeros(self.n)
        self.x, self.y, self.hx, self.hy, self.edad, self.vida = z(), z(), z(), z(), z(), z()
        self.rastro = np.zeros((self.n, self.l, 4))
        self.cabeza = np.zeros(self.n, dtype=int)
        self.largo = np.zeros(self.n, dtype=int)
        self.pasos = 0
        if self.n and self.prob is not None:
            self.nacer(np.arange(self.n))
            self.edad = self.rng.random(self.n) * self.vida
            for _ in range(self.l * 2 + 30):
                self.avanzar()

    def nacer(self, i):
        if not len(i) or self.prob is None:
            return
        celda = self.rng.choice(len(self.prob), size=len(i), p=self.prob)
        self.x[i] = (celda % COLS + self.rng.random(len(i))) * CELDA
        self.y[i] = (celda // COLS + self.rng.random(len(i))) * CELDA
        fa, fb = self.c["fuerza"][0][celda] ** 2, self.c["fuerza"][1][celda] ** 2
        k = (self.rng.random(len(i)) * (fa + fb) >= fa).astype(int)
        self.hx[i] = np.where(k, self.c["ux"][1][celda], self.c["ux"][0][celda])
        self.hy[i] = np.where(k, self.c["uy"][1][celda], self.c["uy"][0][celda])
        self.edad[i], self.vida[i], self.largo[i] = 0, 60 + self.rng.random(len(i)) * 90, 0

    def muestrear(self):
        gx, gy = self.x / CELDA - 0.5, self.y / CELDA - 0.5
        c0, f0 = np.floor(gx).astype(int), np.floor(gy).astype(int)
        tx, ty = gx - c0, gy - f0
        vx, vy, fuerza, rapidez = (np.zeros(self.n) for _ in range(4))
        for dc in (0, 1):
            for df in (0, 1):
                i = np.clip(f0 + df, 0, FILAS - 1) * COLS + np.clip(c0 + dc, 0, COLS - 1)
                peso = (tx if dc else 1 - tx) * (ty if df else 1 - ty)
                rapidez += peso * self.c["rapidez"][i]
                # En cada celda, la corriente que más se parece a la marcha; a contramano no cuenta.
                parecidos = [np.where(self.c["fuerza"][k][i] > 0, self.c["ux"][k][i] * self.hx + self.c["uy"][k][i] * self.hy, 0)
                             for k in (0, 1)]
                mejor = (parecidos[1] > parecidos[0]).astype(int)
                vale = np.maximum(parecidos[0], parecidos[1]) > 0
                fz = np.where(mejor, self.c["fuerza"][1][i], self.c["fuerza"][0][i]) * vale
                vx += peso * fz * np.where(mejor, self.c["ux"][1][i], self.c["ux"][0][i])
                vy += peso * fz * np.where(mejor, self.c["uy"][1][i], self.c["uy"][0][i])
                fuerza += peso * fz
        return vx, vy, fuerza, rapidez

    def avanzar(self):
        if not self.n or self.prob is None:
            return
        self.nacer(np.flatnonzero(self.edad >= self.vida))
        vx, vy, fuerza, rapidez = self.muestrear()
        calma = fuerza < 0.02
        self.vida = np.where(calma, np.minimum(self.vida, self.edad + 10), self.vida)
        m = ~calma
        largo = np.hypot(vx, vy)
        largo[largo == 0] = 1
        hx = self.hx + (vx / largo - self.hx) * 0.5
        hy = self.hy + (vy / largo - self.hy) * 0.5
        h = np.hypot(hx, hy)
        h[h == 0] = 1
        hx, hy = hx / h, hy / h
        paso = 0.45 * (0.22 + 1.9 * rapidez ** 1.6)
        x, y = self.x + hx * paso, self.y + hy * paso
        e = CELDA * 0.75
        I = self.c["intensidad"]
        gx = (bilineal(I, x + e, y) - bilineal(I, x - e, y)) / (2 * e)
        gy = (bilineal(I, x, y + e) - bilineal(I, x, y - e)) / (2 * e)
        proyeccion = gx * hx + gy * hy
        px, py = gx - proyeccion * hx, gy - proyeccion * hy
        empuje = np.hypot(px, py)
        with np.errstate(divide="ignore", invalid="ignore"):
            deriva = np.where(empuje > 0, np.minimum(empuje * self.cenido, paso * 0.6) / empuje, 0)
        x, y = x + px * deriva, y + py * deriva
        fuera = (x < 0) | (x > 120) | (y < 0) | (y > 80)
        self.vida = np.where(m & fuera, np.minimum(self.vida, self.edad + 6), self.vida)
        self.x, self.y = np.where(m, np.clip(x, 0, 120), self.x), np.where(m, np.clip(y, 0, 80), self.y)
        self.hx, self.hy = np.where(m, hx, self.hx), np.where(m, hy, self.hy)
        # La estela guarda un punto nuevo cada dos pasos y entre medias solo mueve la punta.
        avanza = (self.pasos % 2 == 0) | (self.largo == 0)
        self.cabeza = np.where(avanza, (self.cabeza + 1) % self.l, self.cabeza)
        self.largo = np.where(avanza, np.minimum(self.l, self.largo + 1), self.largo)
        self.rastro[np.arange(self.n), self.cabeza] = np.stack([self.x, self.y, rapidez, fuerza], axis=1)
        self.edad += 1
        self.pasos += 1

    def segmentos(self):
        """Los tramos de estela a dibujar: segmentos, colores RGBA y grosores."""
        if not self.n:
            return np.zeros((0, 2, 2)), np.zeros((0, 4)), np.zeros(0)
        k = np.arange(self.l)
        orden = (self.cabeza[:, None] - self.l + 1 + k[None, :]) % self.l
        puntos = self.rastro[np.arange(self.n)[:, None], orden]
        validos = k[None, :] >= self.l - self.largo[:, None]
        apagado = np.minimum(1, self.edad / 8) * np.clip((self.vida - self.edad) / 12, 0, 1)
        alfa = (k[None, 1:] / (self.l - 1)) * apagado[:, None] * 0.8
        usar = validos[:, 1:] & validos[:, :-1] & (alfa >= 0.05)
        seg = np.stack([puntos[:, :-1, :2], puntos[:, 1:, :2]], axis=2)[usar]
        color = RAMPA[np.minimum(6, (puntos[:, 1:, 2][usar] * 7).astype(int))]
        fz = puntos[:, 1:, 3][usar]
        grosor = GROSORES[np.where(fz < 0.34, 0, np.where(fz < 0.67, 1, 2))]
        return seg, np.column_stack([color, alfa[usar]]), grosor


def miles(valor):
    return f"{valor:,.0f}".replace(",", ".")


def archivo_de(texto):
    return "".join(c if c.isalnum() else "_" for c in sin_tildes(texto)).strip("_")


def buscar(nombre):
    for c in campos:
        if sin_tildes(nombre) in sin_tildes(c["nombre"]):
            return c
    raise ValueError(f"No encuentro «{nombre}». Equipos: {', '.join(c['nombre'] for c in campos)}")

In [ ]:
# Todos los equipos de la liga, en una imagen fija (la estela de unos segundos de animación).
columnas = 4 if len(campos) > 6 else 3
filas_fig = int(np.ceil(len(campos) / columnas))
fig, ejes = plt.subplots(filas_fig, columnas, figsize=(4.2 * columnas, 3.1 * filas_fig), facecolor="#0b1013")
for ax in np.ravel(ejes):
    ax.axis("off")
for ax, campo in zip(np.ravel(ejes), sorted(campos, key=lambda c: sin_tildes(c["nombre"]))):
    dibujar_cancha(ax)
    sistema = Particulas(campo)
    for _ in range(40):
        sistema.avanzar()
    seg, color, grosor = sistema.segmentos()
    ax.add_collection(LineCollection(seg, colors=color, linewidths=grosor, capstyle="butt"))
    fila = tabla.loc[campo["nombre"]]
    ax.set_title(f"{campo['nombre']}\n{miles(fila['Balón por partido (m)'])} m/partido · {fila['Hacia delante %']:.0f}% adelante",
                 color="#eaf2f0", fontsize=9, loc="left")
fig.suptitle(f"Flujo de posesión · {NOMBRE_LIGA}   (ataque →, color = velocidad del balón, grosor = uso de la ruta)",
             color="#eaf2f0", fontsize=11, x=0.01, ha="left")
fig.tight_layout()
imagen = SALIDA / "flujo_todos.png"
fig.savefig(imagen, dpi=160, facecolor=fig.get_facecolor())
print(f"Guardado en {imagen}")
plt.show()

## 6 · La animación: dos equipos lado a lado

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

CUADROS = 120                        # 5 s a 24 cuadros por segundo
dos = [buscar(EQUIPO_A), buscar(EQUIPO_B)]
fig, ejes = plt.subplots(1, 2, figsize=(13, 5.2), facecolor="#0b1013")
sistemas, capas = [], []
for ax, campo in zip(ejes, dos):
    dibujar_cancha(ax)
    fila = tabla.loc[campo["nombre"]]
    ax.set_title(f"{campo['nombre']}  ·  {miles(fila['Balón por partido (m)'])} m de balón por partido  ·  "
                 f"{fila['Hacia delante %']:.0f}% hacia delante", color="#eaf2f0", fontsize=10, loc="left")
    sistemas.append(Particulas(campo))
    capas.append(ax.add_collection(LineCollection([], capstyle="butt")))
fig.text(0.01, 0.03, f"{NOMBRE_LIGA} · ataque →  · color: velocidad del balón de {escala['lenta']:.1f} m/s (verde apagado) "
         f"a {escala['rapida']:.1f} m/s (casi blanco) · grosor: cuánto se usa la ruta, en la escala de la liga · Datos: StatsBomb",
         color="#8fa3a0", fontsize=7.5)
fig.subplots_adjust(left=0.01, right=0.99, top=0.9, bottom=0.08, wspace=0.03)


def cuadro(_):
    for sistema, capa in zip(sistemas, capas):
        sistema.avanzar()
        seg, color, grosor = sistema.segmentos()
        capa.set_segments(seg)
        capa.set_color(color)
        capa.set_linewidth(grosor)
    return capas


animacion = FuncAnimation(fig, cuadro, frames=CUADROS, interval=1000 / 24, blit=True)
gif = SALIDA / f"flujo_{archivo_de(dos[0]['nombre'])}_vs_{archivo_de(dos[1]['nombre'])}.gif"
animacion.save(gif, writer=PillowWriter(fps=24), dpi=75, savefig_kwargs={"facecolor": fig.get_facecolor()})
plt.close(fig)
print(f"Guardado en {gif}")
try:
    from IPython.display import Image, display
    display(Image(filename=str(gif)))
except ImportError:
    pass